**This notebook is an exercise in the [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/categorical-variables).**

---


By encoding **categorical variables**, you'll obtain your best results thus far!

# Setup

The questions below will give you feedback on your work. Run the following cell to set up the feedback system.

In [2]:
# Set up code checking
import os
if not os.path.exists("../input/train.csv"):
    os.symlink("../input/home-data-for-ml-course/train.csv", "../input/train.csv")  
    os.symlink("../input/home-data-for-ml-course/test.csv", "../input/test.csv") 
from learntools.core import binder
binder.bind(globals())
from learntools.ml_intermediate.ex3 import *
print("Setup Complete")

Setup Complete


In this exercise, you will work with data from the [Housing Prices Competition for Kaggle Learn Users](https://www.kaggle.com/c/home-data-for-ml-course). 

![Ames Housing dataset image](https://storage.googleapis.com/kaggle-media/learn/images/lTJVG4e.png)

Run the next code cell without changes to load the training and validation sets in `X_train`, `X_valid`, `y_train`, and `y_valid`.  The test set is loaded in `X_test`.

In [135]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
X = pd.read_csv('../input/train.csv', index_col='Id') 
X_test = pd.read_csv('../input/test.csv', index_col='Id')

# Remove rows with missing target, separate target from predictors
X.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X.SalePrice
X.drop(['SalePrice'], axis=1, inplace=True)

# To keep things simple, we'll drop columns with missing values
cols_with_missing = [col for col in X.columns if X[col].isnull().any()] 
X.drop(cols_with_missing, axis=1, inplace=True)
X_test.drop(cols_with_missing, axis=1, inplace=True)

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)

Use the next code cell to print the first five rows of the data.

In [4]:
X_train.head()

,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
619,20,RL,11694,Pave,Reg,Lvl,AllPub,Inside,Gtl,NridgHt,...,108,0,0,260,0,0,7,2007,New,Partial
871,20,RL,6600,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,...,0,0,0,0,0,0,8,2009,WD,Normal
93,30,RL,13360,Pave,IR1,HLS,AllPub,Inside,Gtl,Crawfor,...,0,44,0,0,0,0,8,2009,WD,Normal
818,20,RL,13265,Pave,IR1,Lvl,AllPub,CulDSac,Gtl,Mitchel,...,59,0,0,0,0,0,7,2008,WD,Normal
303,20,RL,13704,Pave,IR1,Lvl,AllPub,Corner,Gtl,CollgCr,...,81,0,0,0,0,0,1,2006,WD,Normal


Notice that the dataset contains both numerical and categorical variables.  You'll need to encode the categorical data before training a model.

To compare different models, you'll use the same `score_dataset()` function from the tutorial.  This function reports the [mean absolute error](https://en.wikipedia.org/wiki/Mean_absolute_error) (MAE) from a random forest model.

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

# Step 1: Drop columns with categorical data

You'll get started with the most straightforward approach.  Use the code cell below to preprocess the data in `X_train` and `X_valid` to remove columns with categorical data.  Set the preprocessed DataFrames to `drop_X_train` and `drop_X_valid`, respectively.  

In [9]:
print(type(X_train.dtypes))
print(X_train.dtypes.head(), len(X_train.dtypes))

<class 'pandas.core.series.Series'>
MSSubClass     int64
MSZoning      object
LotArea        int64
Street        object
LotShape      object
dtype: object 60


In [13]:
# Get list of categorical variables
s = (X_train.dtypes == 'object')
print(type(s))
print(s.head(), len(s))
print(s.head().index)

<class 'pandas.core.series.Series'>
MSSubClass    False
MSZoning       True
LotArea       False
Street         True
LotShape       True
dtype: bool 60
Index(['MSSubClass', 'MSZoning', 'LotArea', 'Street', 'LotShape'], dtype='object')


In [14]:
print(type(s[s]))

<class 'pandas.core.series.Series'>


In [18]:
object_cols = list(s[s].index)

print(f"Categorical variables ({len(object_cols)}):")
print(object_cols)

Categorical variables (27):
['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']


In [19]:
# Fill in the lines below: drop columns in training and validation data
drop_X_train = X_train.loc[:,X_train.dtypes!="object"]
drop_X_valid = X_valid.loc[:,X_valid.dtypes!="object"]

# Check your answers
step_1.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [15]:
# Lines below will give you a hint or solution code
step_1.hint()
step_1.solution()

<IPython.core.display.Javascript object>

<span style="color:#3366cc">Hint:</span> Use the [`select_dtypes()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html) method to drop all columns with the `object` dtype.

<IPython.core.display.Javascript object>

<span style="color:#33cc99">Solution:</span> 
```python
# Drop columns in training and validation data
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

```

Run the next code cell to get the MAE for this approach.

In [20]:
print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop categorical variables):
17837.82570776256


Before jumping into ordinal encoding, we'll investigate the dataset.  Specifically, we'll look at the `'Condition2'` column.  The code cell below prints the unique entries in both the training and validation sets.

In [21]:
print("Unique values in 'Condition2' column in training data:", X_train['Condition2'].unique())
print("\nUnique values in 'Condition2' column in validation data:", X_valid['Condition2'].unique())

Unique values in 'Condition2' column in training data: ['Norm' 'PosA' 'Feedr' 'PosN' 'Artery' 'RRAe']

Unique values in 'Condition2' column in validation data: ['Norm' 'RRAn' 'RRNn' 'Artery' 'Feedr' 'PosN']


# Step 2: Ordinal encoding

### Part A

If you now write code to: 
- fit an ordinal encoder to the training data, and then 
- use it to transform both the training and validation data, 

you'll get an error.  Can you see why this is the case?  (_You'll need  to use the above output to answer this question._)

The training data does not contain entry (e.g. `RRAn`) of an ordinal column that exists in the validation data. That's why the encoder fitted in training data fails to encode the validation data.

In [26]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
# Make copy to avoid changing original data 
label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])

ValueError: Found unknown categories ['RRNn', 'RRAn'] in column 9 during transform

In [27]:
# Check your answer (Run this code cell to receive credit!)
step_2.a.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct:</span> 

Fitting an ordinal encoder to a column in the training data creates a corresponding integer-valued label for each unique value **that appears in the training data**. In the case that the validation data contains values that don't also appear in the training data, the encoder will throw an error, because these values won't have an integer assigned to them.  Notice that the `'Condition2'` column in the validation data contains the values `'RRAn'` and `'RRNn'`, but these don't appear in the training data -- thus, if we try to use an ordinal encoder with scikit-learn, the code will throw an error.

In [28]:
step_2.a.hint()

<IPython.core.display.Javascript object>

<span style="color:#3366cc">Hint:</span> Are there any values that appear in the validation data but not in the training data?

This is a common problem that you'll encounter with real-world data, and there are many approaches to fixing this issue.  For instance, you can write a custom ordinal encoder to deal with new categories.  The simplest approach, however, is to drop the problematic categorical columns.  

Run the code cell below to save the problematic columns to a Python list `bad_label_cols`.  Likewise, columns that can be safely ordinal encoded are stored in `good_label_cols`.

In [22]:
# Categorical columns in the training data
object_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]

# Columns that can be safely ordinal encoded
good_label_cols = [col for col in object_cols if 
                   set(X_valid[col]).issubset(set(X_train[col]))]
        
# Problematic columns that will be dropped from the dataset
bad_label_cols = list(set(object_cols)-set(good_label_cols))
        
print('Categorical columns that will be ordinal encoded:', good_label_cols)
print('\nCategorical columns that will be dropped from the dataset:', bad_label_cols)

Categorical columns that will be ordinal encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'PavedDrive', 'SaleType', 'SaleCondition']

Categorical columns that will be dropped from the dataset: ['Functional', 'Condition2', 'RoofMatl']


### Part B

Use the next code cell to ordinal encode the data in `X_train` and `X_valid`.  Set the preprocessed DataFrames to `label_X_train` and `label_X_valid`, respectively.  
- We have provided code below to drop the categorical columns in `bad_label_cols` from the dataset. 
- You should ordinal encode the categorical columns in `good_label_cols`.  

In [82]:
from sklearn.preprocessing import OrdinalEncoder

# Drop categorical columns that will not be encoded
label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_valid = X_valid.drop(bad_label_cols, axis=1)

# Apply ordinal encoder 
ordinal_encoder = OrdinalEncoder() # Your code here
label_X_train[good_label_cols] = ordinal_encoder.fit_transform(label_X_train[good_label_cols])
label_X_valid[good_label_cols] = ordinal_encoder.transform(label_X_valid[good_label_cols])

# Check your answer
step_2.b.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [83]:
# Lines below will give you a hint or solution code
step_2.b.hint()
step_2.b.solution()

<IPython.core.display.Javascript object>

<span style="color:#3366cc">Hint:</span> Use the `OrdinalEncoder` class from scikit-learn. You should only encode the columns in `good_label_cols`.

<IPython.core.display.Javascript object>

<span style="color:#33cc99">Solution:</span> 
```python
# Drop categorical columns that will not be encoded
label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_valid = X_valid.drop(bad_label_cols, axis=1)

# Apply ordinal encoder
ordinal_encoder = OrdinalEncoder()
label_X_train[good_label_cols] = ordinal_encoder.fit_transform(X_train[good_label_cols])
label_X_valid[good_label_cols] = ordinal_encoder.transform(X_valid[good_label_cols])

```

#### Manual Ordinal Encoding

In [88]:
print(label_X_train.head()[good_label_cols])
print(X_train[good_label_cols[0]].value_counts())
print(type(ordinal_encoder.categories_))
print(ordinal_encoder.categories_[:3])

     MSZoning  Street  LotShape  LandContour  Utilities  LotConfig  LandSlope  \
Id                                                                              
619       3.0     1.0       3.0          3.0        0.0        4.0        0.0   
871       3.0     1.0       3.0          3.0        0.0        4.0        0.0   
93        3.0     1.0       0.0          1.0        0.0        4.0        0.0   
818       3.0     1.0       0.0          3.0        0.0        1.0        0.0   
303       3.0     1.0       0.0          3.0        0.0        0.0        0.0   

     Neighborhood  Condition1  BldgType  ...  ExterQual  ExterCond  \
Id                                       ...                         
619          16.0         2.0       0.0  ...        0.0        4.0   
871          12.0         4.0       0.0  ...        3.0        4.0   
93            6.0         2.0       0.0  ...        3.0        2.0   
818          11.0         2.0       0.0  ...        2.0        4.0   
303         

In [90]:
order = ['C (all)', 'FV', 'RH', 'RL', 'RM']

In [91]:
list(enumerate(order))

[(0, 'C (all)'), (1, 'FV'), (2, 'RH'), (3, 'RL'), (4, 'RM')]

In [93]:
tmp = dict(map(lambda x: reversed(x), enumerate(order)))
tmp

{'C (all)': 0, 'FV': 1, 'RH': 2, 'RL': 3, 'RM': 4}

In [103]:
label_X_train[good_label_cols[0]] = X_train[good_label_cols[0]]
label_X_train[good_label_cols].head()

,MSZoning,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,BldgType,...,ExterQual,ExterCond,Foundation,Heating,HeatingQC,CentralAir,KitchenQual,PavedDrive,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
619,RL,1.0,3.0,3.0,0.0,4.0,0.0,16.0,2.0,0.0,...,0.0,4.0,2.0,1.0,0.0,1.0,2.0,2.0,6.0,5.0
871,RL,1.0,3.0,3.0,0.0,4.0,0.0,12.0,4.0,0.0,...,3.0,4.0,1.0,1.0,2.0,0.0,3.0,2.0,8.0,4.0
93,RL,1.0,0.0,1.0,0.0,4.0,0.0,6.0,2.0,0.0,...,3.0,2.0,0.0,1.0,0.0,1.0,3.0,2.0,8.0,4.0
818,RL,1.0,0.0,3.0,0.0,1.0,0.0,11.0,2.0,0.0,...,2.0,4.0,2.0,1.0,0.0,1.0,2.0,2.0,8.0,4.0
303,RL,1.0,0.0,3.0,0.0,0.0,0.0,5.0,2.0,0.0,...,2.0,4.0,2.0,1.0,0.0,1.0,2.0,2.0,8.0,4.0


In [105]:
label_X_train[good_label_cols[0]] = list(map(lambda x: tmp[x], X_train[good_label_cols[0]]))
label_X_train[good_label_cols].head()

,MSZoning,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,BldgType,...,ExterQual,ExterCond,Foundation,Heating,HeatingQC,CentralAir,KitchenQual,PavedDrive,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
619,3,1.0,3.0,3.0,0.0,4.0,0.0,16.0,2.0,0.0,...,0.0,4.0,2.0,1.0,0.0,1.0,2.0,2.0,6.0,5.0
871,3,1.0,3.0,3.0,0.0,4.0,0.0,12.0,4.0,0.0,...,3.0,4.0,1.0,1.0,2.0,0.0,3.0,2.0,8.0,4.0
93,3,1.0,0.0,1.0,0.0,4.0,0.0,6.0,2.0,0.0,...,3.0,2.0,0.0,1.0,0.0,1.0,3.0,2.0,8.0,4.0
818,3,1.0,0.0,3.0,0.0,1.0,0.0,11.0,2.0,0.0,...,2.0,4.0,2.0,1.0,0.0,1.0,2.0,2.0,8.0,4.0
303,3,1.0,0.0,3.0,0.0,0.0,0.0,5.0,2.0,0.0,...,2.0,4.0,2.0,1.0,0.0,1.0,2.0,2.0,8.0,4.0


In [108]:
label_X_train.dtypes[good_label_cols[:2]]

['MSZoning', 'Street']


MSZoning      int64
Street      float64
dtype: object

In [109]:
# Check your answer
step_2.b.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

Run the next code cell to get the MAE for this approach.

In [27]:
print("MAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

MAE from Approach 2 (Ordinal Encoding):
17098.01649543379


So far, you've tried two different approaches to dealing with categorical variables.  And, you've seen that encoding categorical data yields better results than removing columns from the dataset.

Soon, you'll try one-hot encoding.  Before then, there's one additional topic we need to cover.  Begin by running the next code cell without changes.  

In [30]:
list(zip(['b', 'a', 'c'], [1,2,3]))

[('b', 1), ('a', 2), ('c', 3)]

In [31]:
dict(zip(['b', 'a', 'c'], [1,2,3]))

{'b': 1, 'a': 2, 'c': 3}

In [34]:
# Get number of unique entries in each column with categorical data
X['Condition2'].nunique()

8

In [35]:
# Get number of unique entries in each column with categorical data
object_nunique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# Print number of unique entries by column, in ascending order
sorted(d.items(), key=lambda x: x[1])

[('Street', 2),
 ('Utilities', 2),
 ('CentralAir', 2),
 ('LandSlope', 3),
 ('PavedDrive', 3),
 ('LotShape', 4),
 ('LandContour', 4),
 ('ExterQual', 4),
 ('KitchenQual', 4),
 ('MSZoning', 5),
 ('LotConfig', 5),
 ('BldgType', 5),
 ('ExterCond', 5),
 ('HeatingQC', 5),
 ('Condition2', 6),
 ('RoofStyle', 6),
 ('Foundation', 6),
 ('Heating', 6),
 ('Functional', 6),
 ('SaleCondition', 6),
 ('RoofMatl', 7),
 ('HouseStyle', 8),
 ('Condition1', 9),
 ('SaleType', 9),
 ('Exterior1st', 15),
 ('Exterior2nd', 16),
 ('Neighborhood', 25)]

In [40]:
list(filter(lambda x: x[1]>10, d.items()))

[('Neighborhood', 25), ('Exterior1st', 15), ('Exterior2nd', 16)]

In [47]:
# List Comprehension
[ i for i in d.items() if i[1]>10 ]

[('Neighborhood', 25), ('Exterior1st', 15), ('Exterior2nd', 16)]

# Step 3: Investigating cardinality

### Part A

The output above shows, for each column with categorical data, the number of unique values in the column.  For instance, the `'Street'` column in the training data has two unique values: `'Grvl'` and `'Pave'`, corresponding to a gravel road and a paved road, respectively.

We refer to the number of unique entries of a categorical variable as the **cardinality** of that categorical variable.  For instance, the `'Street'` variable has cardinality 2.

Use the output above to answer the questions below.

In [44]:
# Fill in the line below: How many categorical variables in the training data
# have cardinality greater than 10?
high_cardinality_numcols = len(list(filter(lambda x: x[1]>10, d.items()))) # 3

# Fill in the line below: How many columns are needed to one-hot encode the 
# 'Neighborhood' variable in the training data?
num_cols_neighborhood = d['Neighborhood'] # 25

# Check your answers
step_3.a.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [45]:
# Lines below will give you a hint or solution code
step_3.a.hint()
step_3.a.solution()

<IPython.core.display.Javascript object>

<span style="color:#3366cc">Hint:</span> To one-hot encode a variable, we need one column for each unique entry.

<IPython.core.display.Javascript object>

<span style="color:#33cc99">Solution:</span> 
```python
# How many categorical variables in the training data
# have cardinality greater than 10?
high_cardinality_numcols = 3

# How many columns are needed to one-hot encode the
# 'Neighborhood' variable in the training data?
num_cols_neighborhood = 25

```

### Part B

For large datasets with many rows, one-hot encoding can greatly expand the size of the dataset.  For this reason, we typically will only one-hot encode columns with relatively low cardinality.  Then, high cardinality columns can either be dropped from the dataset, or we can use ordinal encoding.

As an example, consider a dataset with 10,000 rows, and containing one categorical column with 100 unique entries.  
- If this column is replaced with the corresponding one-hot encoding, how many entries are added to the dataset?  
- If we instead replace the column with the ordinal encoding, how many entries are added?  

Use your answers to fill in the lines below.

In [50]:
# Fill in the line below: How many entries are added to the dataset by 
# replacing the column with a one-hot encoding?
OH_entries_added = (100-1)*10000
# the 1 is because the column being replaced is removed

# Fill in the line below: How many entries are added to the dataset by
# replacing the column with an ordinal encoding?
label_entries_added = 1-1

# Check your answers
step_3.b.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [49]:
# Lines below will give you a hint or solution code
step_3.b.hint()
step_3.b.solution()

<IPython.core.display.Javascript object>

<span style="color:#3366cc">Hint:</span> To calculate how many entries are added to the dataset through the one-hot encoding, begin by calculating how many entries are needed to encode the categorical variable (by multiplying the number of rows by the number of columns in the one-hot encoding). Then, to obtain how many entries are **added** to the dataset, subtract the number of entries in the original column.

<IPython.core.display.Javascript object>

<span style="color:#33cc99">Solution:</span> 
```python
# How many entries are added to the dataset by
# replacing the column with a one-hot encoding?
OH_entries_added = 1e4*100 - 1e4

# How many entries are added to the dataset by
# replacing the column with an ordinal encoding?
label_entries_added = 0

```

Next, you'll experiment with one-hot encoding.  But, instead of encoding all of the categorical variables in the dataset, you'll only create a one-hot encoding for columns with cardinality less than 10.

Run the code cell below without changes to set `low_cardinality_cols` to a Python list containing the columns that will be one-hot encoded.  Likewise, `high_cardinality_cols` contains a list of categorical columns that will be dropped from the dataset.

In [51]:
# Columns that will be one-hot encoded
low_cardinality_cols = [col for col in object_cols if X_train[col].nunique() < 10]

# Columns that will be dropped from the dataset
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

print('Categorical columns that will be one-hot encoded:', low_cardinality_cols)
print('\nCategorical columns that will be dropped from the dataset:', high_cardinality_cols)

Categorical columns that will be one-hot encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']

Categorical columns that will be dropped from the dataset: ['Exterior1st', 'Exterior2nd', 'Neighborhood']


# Step 4: One-hot encoding

Use the next code cell to one-hot encode the data in `X_train` and `X_valid`.  Set the preprocessed DataFrames to `OH_X_train` and `OH_X_valid`, respectively.  
- The full list of categorical columns in the dataset can be found in the Python list `object_cols`.
- You should only one-hot encode the categorical columns in `low_cardinality_cols`.  All other categorical columns should be dropped from the dataset. 

Note: refer to https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

`OneHotEncoder`

- We set `handle_unknown='ignore'` to avoid errors when the validation data contains classes that aren't represented in the training data (**handle_unknown** : {‘error’, ‘ignore’, ‘infrequent_if_exist’, ‘warn’}, default=’error’)
    - ‘ignore’ : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.
- setting `sparse_output=False` ensures that the encoded columns are returned as a numpy array (instead of a sparse matrix).

In [112]:
from sklearn.preprocessing import OneHotEncoder

# Use as many lines of code as you need!

# OH_X_train = ____ # Your code here
# OH_X_valid = ____ # Your code here

# Check your answer
# step_4.check()

In [113]:
# Lines below will give you a hint or solution code
step_4.hint()
step_4.solution()

<IPython.core.display.Javascript object>

<span style="color:#3366cc">Hint:</span> Begin by applying the one-hot encoder to the low cardinality columns in the training and validation data in `X_train[low_cardinality_cols]` and `X_valid[low_cardinality_cols]`, respectively.

<IPython.core.display.Javascript object>

<span style="color:#33cc99">Solution:</span> 
```python
# Apply one-hot encoder to each column with categorical data
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols]))

# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

```

In [122]:
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols]), index=X_train.index)
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols]))

print(OH_cols_train.head())
OH_cols_valid.head()

     0    1    2    3    4    5    6    7    8    9    ...  112  113  114  \
Id                                                     ...                  
619  0.0  0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0  0.0  ...  0.0  1.0  0.0   
871  0.0  0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
93   0.0  0.0  0.0  1.0  0.0  0.0  1.0  1.0  0.0  0.0  ...  0.0  0.0  0.0   
818  0.0  0.0  0.0  1.0  0.0  0.0  1.0  1.0  0.0  0.0  ...  0.0  0.0  0.0   
303  0.0  0.0  0.0  1.0  0.0  0.0  1.0  1.0  0.0  0.0  ...  0.0  0.0  0.0   

     115  116  117  118  119  120  121  
Id                                      
619  0.0  0.0  0.0  0.0  0.0  0.0  1.0  
871  1.0  0.0  0.0  0.0  0.0  1.0  0.0  
93   1.0  0.0  0.0  0.0  0.0  1.0  0.0  
818  1.0  0.0  0.0  0.0  0.0  1.0  0.0  
303  1.0  0.0  0.0  0.0  0.0  1.0  0.0  

[5 rows x 122 columns]


,0,1,2,3,4,5,6,7,8,9,...,112,113,114,115,116,117,118,119,120,121
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [123]:
OH_cols_valid.index = X_valid.index

In [125]:
num_X_train=X_train.drop(object_cols, axis=1)
num_X_valid=X_valid.drop(object_cols, axis=1)

print(num_X_train.shape, num_X_valid.shape)

num_X_train.head()

(1168, 33) (292, 33)


,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
Id,,,,,,,,,,,,,,,,,,,,,
619,20,11694,9,5,2007,2007,48,0,1774,1822,...,774,0,108,0,0,260,0,0,7,2007
871,20,6600,5,5,1962,1962,0,0,894,894,...,308,0,0,0,0,0,0,0,8,2009
93,30,13360,5,7,1921,2006,713,0,163,876,...,432,0,0,44,0,0,0,0,8,2009
818,20,13265,8,5,2002,2002,1218,0,350,1568,...,857,150,59,0,0,0,0,0,7,2008
303,20,13704,7,5,2001,2002,0,0,1541,1541,...,843,468,81,0,0,0,0,0,1,2006


In [127]:
OH_X_train = num_X_train.join(OH_cols_train, how='outer')
OH_X_train.head()

,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,112,113,114,115,116,117,118,119,120,121
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,2003,706,0,150,856,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,70,9550,7,5,1915,1970,216,0,540,756,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
7,20,10084,8,5,2004,2005,1369,0,317,1686,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
8,60,10382,7,6,1973,1973,859,32,216,1107,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
9,50,6120,7,5,1931,1950,0,0,952,952,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0


In [128]:
# OH_X_train = ____ # Your code here high_cardinality_cols
# OH_X_valid = ____ # Your code here
OH_X_valid = num_X_valid.join(OH_cols_valid, how='outer')

# Check your answer
step_4.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

Run the next code cell to get the MAE for this approach.

In [131]:
# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

# Without this, the next cell gives:

TypeError: Feature names are only supported if all input features have string names, but your input has ['int', 'str'] as feature name / column name types. If you want feature names to be stored and validated, you must convert them all to strings, by using X.columns = X.columns.astype(str) for example. Otherwise you can remove feature / column names from your input data, or convert them all to a non-string data type.

In [132]:
print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

MAE from Approach 3 (One-Hot Encoding):
60289.05045662101


# Generate test predictions and submit your results

After you complete Step 4, if you'd like to use what you've learned to submit your results to the leaderboard, you'll need to preprocess the test data before generating predictions.

**This step is completely optional, and you do not need to submit results to the leaderboard to successfully complete the exercise.**

Check out the previous exercise if you need help with remembering how to [join the competition](https://www.kaggle.com/c/home-data-for-ml-course) or save your results to CSV.  Once you have generated a file with your results, follow the instructions below:
1. Begin by clicking on the **Save Version** button in the top right corner of the window.  This will generate a pop-up window.  
2. Ensure that the **Save and Run All** option is selected, and then click on the **Save** button.
3. This generates a window in the bottom left corner of the notebook.  After it has finished running, click on the number to the right of the **Save Version** button.  This pulls up a list of versions on the right of the screen.  Click on the ellipsis **(...)** to the right of the most recent version, and select **Open in Viewer**.  This brings you into view mode of the same page. You will need to scroll down to get back to these instructions.
4. Click on the **Data** tab near the top of the screen.  Then, click on the file you would like to submit, and click on the **Submit** button to submit your results to the leaderboard.

You have now successfully submitted to the competition!

If you want to keep working to improve your performance, select the **Edit** button in the top right of the screen. Then you can change your code and repeat the process. There's a lot of room to improve, and you will climb up the leaderboard as you work.


## Last Submission

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
# Read the data
X_full = pd.read_csv('../input/train.csv', index_col='Id')
X_test_full = pd.read_csv('../input/test.csv', index_col='Id')

# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# To keep things simple, we'll use only numerical predictors
X = X_full.select_dtypes(exclude=['object'])
X_test = X_test_full.select_dtypes(exclude=['object'])

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,
                                                      random_state=0)

In [3]:
# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [4]:
# (Optional) Your code here
def preprocess_X(df_X):
    ret = df_X.drop(['GarageYrBlt'], axis=1)

    masvnrNullIdx=ret.MasVnrArea.isnull()
    ret.loc[masvnrNullIdx, "MasVnrArea"]=103.5 # 103.5: (2) 17809.983767123285, 0: (3) 17929.69297945206
    # ret['MasVnrArea_empty']=masvnrNullIdx # Removing this yielded (1) 18105.497933789953 => (2) 17809.983767123285

    lotFrontageNullIdx = ret.LotFrontage.isnull()
    ret.loc[lotFrontageNullIdx, "LotFrontage"] = 69
    # ret['LotFrontage_empty']=lotFrontageNullIdx # Removing this yielded (1) 18105.497933789953 => (2) 17809.983767123285
    
    return ret

In [5]:
# Preprocessed training and validation features
final_X_train = preprocess_X(X_train)
final_X_valid = preprocess_X(X_valid)

# Define and fit model
model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(final_X_train, y_train)

# Get validation predictions and MAE
preds_valid = model.predict(final_X_valid)
print("MAE (Your approach):")
print(mean_absolute_error(y_valid, preds_valid))

MAE (Your approach):
17809.983767123285


## Experiment Setup

In [74]:
# Read the data
X = pd.read_csv('../input/train.csv', index_col='Id')
X_test = pd.read_csv('../input/test.csv', index_col='Id')

# Remove rows with missing target, separate target from predictors
X.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X.SalePrice
X.drop(['SalePrice'], axis=1, inplace=True)

In [75]:
print(type(X.dtypes))
print(X.dtypes.head(), len(X.dtypes))

<class 'pandas.core.series.Series'>
MSSubClass       int64
MSZoning        object
LotFrontage    float64
LotArea          int64
Street          object
dtype: object 79


In [76]:
# Get list of categorical variables
s = (X.dtypes == 'object')
print(type(s))
print(s.head(), len(s))
print(s.head().index)

<class 'pandas.core.series.Series'>
MSSubClass     False
MSZoning        True
LotFrontage    False
LotArea        False
Street          True
dtype: bool 79
Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street'], dtype='object')


In [77]:
print(sum(s))
objCols = X_train.columns[s]
print(sorted(objCols))

43
['Alley', 'BldgType', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'CentralAir', 'Condition1', 'Condition2', 'Electrical', 'ExterCond', 'ExterQual', 'Exterior1st', 'Exterior2nd', 'Fence', 'FireplaceQu', 'Foundation', 'Functional', 'GarageCond', 'GarageFinish', 'GarageQual', 'GarageType', 'Heating', 'HeatingQC', 'HouseStyle', 'KitchenQual', 'LandContour', 'LandSlope', 'LotConfig', 'LotShape', 'MSZoning', 'MasVnrType', 'MiscFeature', 'Neighborhood', 'PavedDrive', 'PoolQC', 'RoofMatl', 'RoofStyle', 'SaleCondition', 'SaleType', 'Street', 'Utilities']


In [78]:
X_full = pd.concat([X, X_test])
print(X.shape, X_test.shape, X_full.shape)

(1460, 79) (1459, 79) (2919, 79)


In [79]:
# To keep things simple, we'll use only numerical predictors
# X = X_full.select_dtypes(exclude=['object'])
# X_test = X_test_full.select_dtypes(exclude=['object'])

# To keep things simple, we'll drop 'object' columns with missing values
obj_cols_with_missing = [col for col in objCols if X_full[col].isnull().any()]
print(len(obj_cols_with_missing))
print(len(objCols))
objCols = list(set(objCols)-set(obj_cols_with_missing))
print(len(objCols))

23
43
20


In [80]:
X.drop(obj_cols_with_missing, axis=1, inplace=True)
X_test.drop(obj_cols_with_missing, axis=1, inplace=True)
X_full = pd.concat([X, X_test])

X_full.shape

(2919, 56)

Finding # Domain...



In [81]:
# Get number of unique entries in each column with categorical data
X_full[objCols].nunique().head()

LandSlope        3
LotConfig        5
Neighborhood    25
Street           2
Condition2       8
dtype: int64

In [82]:
# Get number of unique entries in each column with categorical data
object_nunique = list(map(lambda col: X_full[col].nunique(), objCols))
d = dict(zip(objCols, object_nunique))

# Print number of unique entries by column, in ascending order
sorted(d.items(), key=lambda x: x[1])

[('Street', 2),
 ('CentralAir', 2),
 ('LandSlope', 3),
 ('PavedDrive', 3),
 ('ExterQual', 4),
 ('LandContour', 4),
 ('LotShape', 4),
 ('LotConfig', 5),
 ('HeatingQC', 5),
 ('BldgType', 5),
 ('ExterCond', 5),
 ('Foundation', 6),
 ('SaleCondition', 6),
 ('RoofStyle', 6),
 ('Heating', 6),
 ('Condition2', 8),
 ('HouseStyle', 8),
 ('RoofMatl', 8),
 ('Condition1', 9),
 ('Neighborhood', 25)]

In [83]:
# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,
                                                      random_state=0)

## Experiment 0 - Plain Ordinal

The 'default', 'Ordinal encode - every readily possible values' approach

Q.(Me) Why not fit_transform the whole X?

A.(ChatGPT) **data leakage**. The encoder would learn categories from validation/test data, which the model should not see during training.

The tutorial tries to avoid unseen-category errors without introducing new parameters. It’s a teaching simplification.

In real ML work, people usually use:

handle_unknown="use_encoded_value" (OrdinalEncoder)
handle_unknown="ignore" (OneHotEncoder)

OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)


In [ ]:
# Categorical columns in the training data
object_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]

# Columns that can be safely ordinal encoded
good_label_cols = [col for col in object_cols if 
                   set(X_valid[col]).issubset(set(X_train[col]))]
        
# Problematic columns that will be dropped from the dataset
bad_label_cols = list(set(object_cols)-set(good_label_cols))

print('Categorical columns that will be ordinal encoded:', good_label_cols)
print('\nCategorical columns that will be dropped from the dataset:', bad_label_cols)

from sklearn.preprocessing import OrdinalEncoder

# Drop categorical columns that will not be encoded
label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_valid = X_valid.drop(bad_label_cols, axis=1)

# Apply ordinal encoder 
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1) # Your code here
label_X_train[good_label_cols] = ordinal_encoder.fit_transform(label_X_train[good_label_cols])
label_X_valid[good_label_cols] = ordinal_encoder.transform(label_X_valid[good_label_cols])

In [ ]:
print("MAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

## Experiment 1 - My Selection

Let's Only consider four more variables now. 'Street' and 'CentralAir', 'LotShape' and 'ExterCond'

Street: Type of road access to property

       Grvl	Gravel	
       Pave	Paved

CentralAir: Central air conditioning

       N	No
       Y	Yes

LotShape: General shape of property

       Reg	Regular	
       IR1	Slightly irregular
       IR2	Moderately Irregular
       IR3	Irregular

ExterCond: Evaluates the present condition of the material on the exterior
		
       Ex	Excellent
       Gd	Good
       TA	Average/Typical
       Fa	Fair
       Po	Poor

## Submission

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer()
imputer.fit_transform(final_X_train)

# Fill in the line below: preprocess test data
final_X_test = pd.DataFrame(imputer.transform(preprocess_X(X_test)))
final_X_test.columns = final_X_train.columns

# Fill in the line below: get test predictions
preds_test = model.predict(final_X_test)

In [ ]:
# Save test predictions to file
output = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': preds_test})
output.to_csv('submission.csv', index=False)

# Keep going

With missing value handling and categorical encoding, your modeling process is getting complex. This complexity gets worse when you want to save your model to use in the future. The key to managing this complexity is something called **pipelines**. 

**[Learn to use pipelines](https://www.kaggle.com/alexisbcook/pipelines)** to preprocess datasets with categorical variables, missing values and any other messiness your data throws at you.

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intermediate-machine-learning/discussion) to chat with other learners.*